In [ ]:
!pip install -q transformers datasets accelerate soundfile librosa jiwer evaluate peft

In [ ]:
import torch
import re
import json
import os
import numpy as np
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Union

from datasets import load_from_disk, Audio
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model
import evaluate

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Set the correct absolute paths to your Kaggle dataset
train_path = "/kaggle/input/datasets/aadityanair1/mls-spanish/mls_spanish_train"
test_path  = "/kaggle/input/datasets/aadityanair1/mls-spanish/mls_spanish_test"

train_dataset = load_from_disk(train_path)
test_dataset  = load_from_disk(test_path)

# Ensure sampling rate is 16 kHz
train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
test_dataset  = test_dataset.cast_column("audio",  Audio(sampling_rate=16_000))

print(f"Train columns : {train_dataset.column_names}")
print(f"Train size    : {len(train_dataset)}")
print(f"Test size     : {len(test_dataset)}")

In [ ]:
chars_to_ignore_regex = r'[\,\.\!\-\;\:\"\n¿¡]'

def extract_all_chars(batch):
    # Find the text column (MLS uses 'transcript'; FLEURS uses 'sentence')
    text_col = next(
        (c for c in ("transcript", "sentence", "text") if c in batch),
        None
    )
    if text_col is None:
        raise ValueError(f"No text column found. Available: {list(batch.keys())}")

    all_text = " ".join(
        re.sub(chars_to_ignore_regex, '', t).lower()
        for t in batch[text_col]
    )
    vocab = list(set(all_text))
    return {"vocab": [vocab], "all_text": [all_text]}

vocabs_train = train_dataset.map(
    extract_all_chars, batched=True, batch_size=-1,
    keep_in_memory=True, remove_columns=train_dataset.column_names
)
vocabs_test = test_dataset.map(
    extract_all_chars, batched=True, batch_size=-1,
    keep_in_memory=True, remove_columns=test_dataset.column_names
)

vocab_list = list(set(vocabs_train["vocab"][0]) | set(vocabs_test["vocab"][0]))
vocab_dict = {v: k for k, v in enumerate(vocab_list)}

# Replace space with pipe (word delimiter for CTC)
if " " in vocab_dict:
    vocab_dict["|"] = vocab_dict.pop(" ")

vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

print(f"Total characters in vocabulary: {len(vocab_dict)}")
print(sorted(vocab_dict.keys()))

In [ ]:
# Save vocab and build tokenizer
with open("vocab.json", "w") as f:
    json.dump(vocab_dict, f)

tokenizer = Wav2Vec2CTCTokenizer(
    "./vocab.json",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|"
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True
)

processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)
processor.save_pretrained("./wav2vec2-xls-r-fleurs-spanish")

print("Processor saved.")
print(f"Vocab size: {len(processor.tokenizer)}")
print(processor.tokenizer.get_vocab())

In [ ]:
def prepare_dataset_for_model(batch):
    audio = batch["audio"]

    # --- Audio ---
    batch["input_values"] = processor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_values[0]

    # --- Text: find whichever column exists ---
    target_text = batch.get("transcript") or batch.get("sentence") or batch.get("text")
    if target_text is None:
        raise ValueError(f"No text column found. Keys: {list(batch.keys())}")

    clean_text = re.sub(chars_to_ignore_regex, '', target_text).lower()
    batch["labels"] = processor(text=clean_text).input_ids
    return batch

columns_to_remove = train_dataset.column_names

# Cache to a writable directory to avoid read-only errors on Kaggle
cache_dir = "/kaggle/working/dataset_cache"
os.makedirs(cache_dir, exist_ok=True)

train_dataset = train_dataset.map(
    prepare_dataset_for_model,
    remove_columns=columns_to_remove,
    batched=False,
    cache_file_name=os.path.join(cache_dir, "train_cache.arrow")
)
test_dataset = test_dataset.map(
    prepare_dataset_for_model,
    remove_columns=test_dataset.column_names,
    batched=False,
    cache_file_name=os.path.join(cache_dir, "test_cache.arrow")
)

print(f"Columns remaining: {train_dataset.column_names}")
print(f"Train size: {len(train_dataset)}")

In [ ]:
sample = train_dataset[0]
print(f"Remaining Columns : {train_dataset.column_names}")
print(f"Sample Label IDs  : {sample['labels'][:10]}")
print(f"Decoded Sample    : {processor.decode(sample['labels'])}")

In [ ]:
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]}          for f in features]

        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")

        labels_batch = self.processor.pad(labels=label_features, padding=self.padding, return_tensors="pt")
        # Replace padding index with -100 so loss ignores it
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

# Quick smoke-test
features = [train_dataset[i] for i in range(2)]
batch = data_collator(features)
print(batch.keys())
print("Labels shape:", batch["labels"].shape)

In [ ]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids    = np.argmax(pred_logits, axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str  = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer, "cer": cer}

print("Metrics ready.")

In [ ]:
model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-xls-r-300m",
    attention_dropout=0.0,
    hidden_dropout=0.0,
    feat_proj_dropout=0.0,
    mask_time_prob=0.05,
    layerdrop=0.0,          # CRITICAL: keep all layers during LoRA training
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
)

peft_config = LoraConfig(
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj"],
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    modules_to_save=["lm_head"],
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# Spec-augment settings
model.config.mask_time_prob    = 0.075
model.config.mask_feature_prob = 0.05
model.config.apply_spec_augment = True
model.config.layerdrop          = 0.0
model.config.ctc_loss_reduction = "mean"

# Fix embedding save bug for PEFT + Wav2Vec2
model.get_input_embeddings  = lambda: model.base_model.model.wav2vec2.feature_extractor
model.get_output_embeddings = lambda: model.base_model.model.lm_head

training_args = TrainingArguments(
    output_dir="./xlsr-spanish-lora",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    fp16=True,
    learning_rate=2e-4,
    num_train_epochs=5,
    warmup_steps=50,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    logging_steps=5,
    report_to="none",
    remove_unused_columns=False,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=processor,
)

trainer.train()

In [ ]:
trainer.save_model("./final-spanish-model")
processor.save_pretrained("./final-spanish-model")
print("Model and processor saved to ./final-spanish-model")